In [ ]:
# Work with Kaggle
# Force CUDA to be 1
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
print(torch.cuda.device_count())  

In [ ]:
!pip install -q --upgrade  bitsandbytes trl

In [ ]:
# Check if we are working on T4
# Check if we are working on T4 (Kaggle)
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)

if gpu_info.find('failed') >= 0 or 'NVIDIA-SMI has failed' in gpu_info:
    print('Not connected to a GPU')
    print("On Kaggle: go to Settings (right sidebar) > Accelerator > GPU T4 x2")
else:
    print('GPU info:')
    print(gpu_info)
    if gpu_info.find('Tesla T4') >= 0:
        print("\nSuccess - Connected to a T4")
        # Kaggle T4 sessions give you 2 GPUs - check how many are visible
        t4_count = gpu_info.count('Tesla T4')
        print(f"Number of T4s detected: {t4_count}")
    else:
        print("\nNOT CONNECTED TO a T4")

In [ ]:
# Fine tuning: tokenization ---> quntization ---> Set hyper-paramters ---> create trainer ---> fine tuning ---> monitoring
# Imports
import sys
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from datetime import datetime
import wandb
from trl import SFTTrainer, SFTConfig

In [ ]:
# Connect with utils folder
import sys
sys.path.append('/kaggle/input/datasets/omarshaarawy/pricer')
# Class Item to validate the returned data and has lots of helper functions
from items import Item

In [ ]:
!git config --global credential.helper store

In [ ]:
# Login to Hugging Face
from huggingface_hub import login
# Library to access Hugging Face token from user secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()         
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
# login to WANDB
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

In [ ]:
# Download the fine-tuning dataset from Hugging Face
username = "omarshaarawy11"
dataset = f"{username}/best_deal_04_fine_tuning"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

In [ ]:
# Resume the training from the last checkpoint
CHECKPOINT = "/kaggle/input/datasets/omarshaarawy/best-deal-resume-checkpoint/best-deal-2026-09-10_18.41.06/checkpoint-480"
WANDB_RUN_ID = "5ui1rjma"

In [ ]:
# Configurations
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# The constant project name
PROJECT_NAME = "best-deal"
USERNAME = "omarshaarawy11"
# RUN_NAME: the run name of current snapshot of project ---> (Hugging Face (partially) and WANDB)
# RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
# Resume the training from last checkpoint
RUN_NAME = "2026-09-10_18.41.06"
# PROJECT_RUN_NAME: for output directory ---> (Hugging Face)
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# Name of our model on Hugging Face
HUB_MODEL_NAME = f"{USERNAME}/{PROJECT_RUN_NAME}"
# Hyper parameters
EPOCHS = 1
# Depends his big on how to fit GPU
BATCH_SIZE = 4
# Remeber we have max tokens we don't wanna exceed which was 90 + additional space for upcoming extra tokens
MAX_SEQUENCE_LENGTH = 128
# When it's one means we upadate weights after 1 step which is the normal flow
GRADIENT_ACCUMULATION_STEPS = 8
# QLoRA paramters
# Quantization to 4 bits
QUANT_4_BIT = True
LORA_R = 4
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS
LORA_DROPOUT = 0.1
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"
# Monitoring
# How much steps to report (log) our metrics to WANDB
LOG_STEPS = 5
# How much steps we take snapshot (check point) and save it on Hugging Face
SAVE_STEPS = 5
LOG_TO_WANDB = True

In [ ]:
#if LOG_TO_WANDB:
#  wandb.init(project=PROJECT_NAME, name=RUN_NAME, entity="omar-team-ai")
# Resume the training from the last checkpoint
if LOG_TO_WANDB:
    wandb.init(
        project=PROJECT_NAME,
        entity="omar-team-ai",
        id=WANDB_RUN_ID,
        resume="must"
    )

In [ ]:
# Quantization configurations
# For 4 bits quantization
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    # Make it float16 when working in Kaggle  
    bnb_4bit_compute_dtype=torch.float16,
    # nf4 stands for normal float 4
    bnb_4bit_quant_type="nf4")
# For 8 bits quantization
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [ ]:
# Tokenization
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Quantization
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

# Prepare the 4-bit model for QLoRA training
base_model = prepare_model_for_kbit_training(base_model)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Check the amount of memory that model occupies in RAM or GPU memory
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

In [ ]:
# Set hyper-paramters
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)


train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,

    # Kaggle T4 supports FP16, not BF16
    fp16=False,
    bf16=False,

    max_grad_norm=0.3,
    max_steps=-1,

    # Use ratio, not steps
    warmup_steps=WARMUP_RATIO,

    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,

    save_strategy="steps",

    # Save checkpoints to Hugging Face
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,

    eval_strategy="steps",
    eval_steps=SAVE_STEPS,

    # Start a completely fresh training run on Kaggle
    resume_from_checkpoint=False,

    # When we run this code inside Kaggle
    loss_type="nll"
)

In [ ]:
# Create trainer
# Convert lists of Item objects to lists of dictionaries with a 'text' field
train_data_dicts = [{
    "text": f"{item.prompt} {item.completion}"
} for item in train]
val_data_dicts = [{
    "text": f"{item.prompt} {item.completion}"
} for item in val]
train_dataset = datasets.Dataset.from_list(train_data_dicts)
val_dataset = datasets.Dataset.from_list(val_data_dicts)
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    # hyper-paramters
    peft_config=lora_parameters,
    args=train_parameters
)

In [ ]:
# Fine-tuning
# Resume the training from last checkpoint
# fine_tuning.train()
fine_tuning.train(resume_from_checkpoint=CHECKPOINT)
# This protect us from losing every thing when Kaggle is collapsed by saving it to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}") 

In [ ]:
# Close monitoring session
if LOG_TO_WANDB:
  wandb.finish()